# Monte Carlo Neutronics of a PWR Fuel Pin with OpenMC and pandas
This notebook builds an infinite-lattice PWR UO₂ pin cell (reflective boundaries), solves the k-eigenvalue
problem with OpenMC's continuous-energy Monte Carlo solver, and analyses the tallies systematically with
pandas DataFrames. It is a modernised, heavily annotated counterpart of the official
`openmc-notebooks/pandas-dataframes.ipynb`, updated for OpenMC ≥ 0.16 (Model API, source constraints).

**Learning goals:** the neutron transport equation → Monte Carlo random walks → power iteration and k_eff →
inactive/active batches and Shannon entropy → tallies, estimators and uncertainties → pandas analysis.

## 1. The neutron transport equation
The angular flux ψ(r, Ω, E) (neutrons·cm⁻²·s⁻¹·sr⁻¹·eV⁻¹) obeys the steady-state Boltzmann equation:

$$\Omega\cdot\nabla\psi + \Sigma_t(\mathbf r,E)\psi = \int_0^\infty\!\!\int_{4\pi}\Sigma_s(\mathbf r,E'\to E,\Omega'\to\Omega)\psi\,d\Omega' dE' + \frac{\chi(E)}{4\pi k}\int_0^\infty \nu\Sigma_f(\mathbf r,E')\phi(\mathbf r,E')\,dE'$$

- Σ_x = N σ_x is a macroscopic cross section (cm⁻¹): atom density N (atoms/b-cm) × microscopic cross section σ_x (barns).
- φ = ∫ψ dΩ is the scalar flux, ν is neutrons per fission, χ(E) is the fission spectrum.
- With no external source this is an **eigenvalue problem**: only the factor 1/k makes a steady state possible.
  k_eff > 1 is supercritical, = 1 critical, < 1 subcritical.

Monte Carlo does not discretise this equation. Instead it simulates individual neutron histories whose
*expected* behaviour satisfies it exactly: distance to collision ~ Exp(Σ_t), the collision nuclide is chosen
∝ Σ_{t,i}, and the reaction ∝ σ_x/σ_t, all with continuous-energy nuclear data.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openmc

print("OpenMC version:", openmc.__version__)

# Cross-section data: OpenMC reads a cross_sections.xml index pointing to HDF5 files.
# openmc.config['cross_sections'] is initialised from $OPENMC_CROSS_SECTIONS at import time.
xs = os.environ.get("OPENMC_CROSS_SECTIONS")
if xs is None:
    # Fallback: edit this path to where tar data is unpacked i.e. the ENDF/B-VIII.0 library from openmc.org/data
    xs = str(Path.home() / "nuclear_data" / "endfb-viii.0-hdf5" / "cross_sections.xml")
openmc.config["cross_sections"] = xs
assert Path(openmc.config["cross_sections"]).exists(), "cross_sections.xml not found - see README"

workdir = Path("run_pincell"); workdir.mkdir(exist_ok=True)

## 2. Nuclear data and cross sections
Evaluated libraries (ENDF/B-VIII.0, JEFF, JENDL) are processed by NJOY into OpenMC's HDF5 format. Each nuclide
file holds pointwise σ(E) at several temperatures (250–2500 K for the official libraries), resonance
probability tables, secondary angle/energy distributions, and ν(E). **Thermal scattering S(α,β)** data
(`c_H_in_H2O`) captures the effect of molecular binding of hydrogen in water below ~4 eV. Leaving it out
(as the original pandas notebook does) visibly distorts the thermal spectrum and k_eff.

In [ ]:
# Composition follows the official examples/pincell/build_xml.py (2.4 wt% UO2, Zircaloy-4, borated water)
uo2 = openmc.Material(name="UO2 fuel 2.4 wt%")
uo2.set_density("g/cm3", 10.29769)
uo2.add_element("U", 1.0, enrichment=2.4)    # enrichment in wt% U-235
uo2.add_element("O", 2.0)
uo2.temperature = 900                        # K, approximate fuel temperature

helium = openmc.Material(name="He gap")
helium.set_density("g/cm3", 0.001598)
helium.add_element("He", 2.4044e-4)

zircaloy = openmc.Material(name="Zircaloy-4")
zircaloy.set_density("g/cm3", 6.55)
zircaloy.add_element("Sn", 0.014, "wo")
zircaloy.add_element("Fe", 0.00165, "wo")
zircaloy.add_element("Cr", 0.001, "wo")
zircaloy.add_element("Zr", 0.98335, "wo")

water = openmc.Material(name="Borated water")
water.set_density("g/cm3", 0.740582)
water.add_element("B", 4.0e-5)
water.add_element("H", 5.0e-2)
water.add_element("O", 2.4e-2)
water.add_s_alpha_beta("c_H_in_H2O")

materials = openmc.Materials([uo2, helium, zircaloy, water])
for m in materials:
    print(m.name, {k: f"{v:.3e}" for k, v in m.get_nuclide_atom_densities().items()})

`add_element` expands natural elements into isotopes using IUPAC abundances, and only keeps isotopes present
in your cross-section library. `get_nuclide_atom_densities()` returns atoms/b-cm, i.e. the N in Σ = Nσ.
Note on temperature: OpenMC interpolates between the library temperatures (default method 'nearest'). See
`Settings.temperature` if you need interpolation.

## 3. Constructive solid geometry (CSG)
Surfaces define half-spaces (−S inside, +S outside). Cells are Boolean combinations of half-spaces filled with
materials. A `RectangularPrism` with `boundary_type='reflective'` mirrors every neutron that reaches the outer
boundary. Physically this models an **infinite lattice of identical pins**, so there is zero net leakage and the
result is k_∞ rather than k_eff of a finite core.